# ExperimentLab — Full Python Code

**Hiring purpose:** one project, one notebook, with the actual Python implementation visible. The modular files remain in the repository because that is how production code should be organised; this notebook mirrors those files so a recruiter can inspect the full code without hunting.


## Dataset and reproducibility

Deterministic synthetic randomized-experiment data with a known treatment effect.

The project README/data card documents provenance, constraints and the exact reproduction path. Large third-party raw files are not duplicated in Git when licensing or repository size makes that poor engineering practice.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_SLUG = 'experiment_lab'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    target = Path('/content/uni_projects')
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Jorgoluka100/uni_projects.git', str(target)], check=True)
    os.chdir(target)
    ROOT = target
PROJECT = ROOT / 'projects' / PROJECT_SLUG
assert PROJECT.exists(), PROJECT
print('Project:', PROJECT.resolve())


## Full Python implementation

Every code cell below is copied directly from the corresponding `.py` file on the same commit. These cells are intentionally tagged `source-mirror` so the notebook acts as a readable code portfolio while the canonical modules remain testable files.


### `run.py`


In [ ]:
from __future__ import annotations

import argparse
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm


def simulate(n: int, seed: int, effect: float) -> pd.DataFrame:
    rng=np.random.default_rng(seed); pre=rng.normal(100,20,n); segment=rng.choice(["new","returning"],n,p=[0.35,0.65]); treatment=rng.integers(0,2,n); noise=rng.normal(0,12,n)
    outcome=25+0.62*pre+4.0*(segment=="returning")+effect*treatment+noise; guardrail=rng.normal(5.0,1.1,n)+0.02*treatment
    return pd.DataFrame({"pre_metric":pre,"segment":segment,"treatment":treatment,"outcome":outcome,"guardrail":guardrail})


def mean_effect(y: np.ndarray,t: np.ndarray)->tuple[float,float,float]:
    yt,yc=y[t==1],y[t==0]; effect=float(yt.mean()-yc.mean()); se=math.sqrt(float(yt.var(ddof=1)/len(yt)+yc.var(ddof=1)/len(yc))); return effect,effect-1.96*se,effect+1.96*se


def cuped_adjust(y:np.ndarray,x:np.ndarray)->tuple[np.ndarray,float]:
    theta=float(np.cov(y,x,ddof=1)[0,1]/np.var(x,ddof=1)); return y-theta*(x-x.mean()),theta


def stratified_bootstrap(y:np.ndarray,t:np.ndarray,rounds:int,seed:int)->tuple[float,float]:
    rng=np.random.default_rng(seed); ti,ci=np.where(t==1)[0],np.where(t==0)[0]; values=[]
    for _ in range(rounds):
        ts=rng.choice(ti,len(ti),replace=True); cs=rng.choice(ci,len(ci),replace=True); values.append(float(y[ts].mean()-y[cs].mean()))
    return tuple(float(x) for x in np.quantile(values,[0.025,0.975]))


def power_and_mde(sd:float,n:int,alpha:float=0.05,power:float=0.80)->dict[str,float]:
    per_arm=n/2; return {"alpha":alpha,"target_power":power,"mde_outcome_units":float((norm.ppf(1-alpha/2)+norm.ppf(power))*sd*math.sqrt(2/per_arm))}


def run(n:int,seed:int,effect:float,output_dir:Path)->dict:
    output_dir.mkdir(parents=True,exist_ok=True); df=simulate(n,seed,effect); y=df.outcome.to_numpy(); t=df.treatment.to_numpy(); pre=df.pre_metric.to_numpy()
    raw=mean_effect(y,t); adjusted,theta=cuped_adjust(y,pre); cuped=mean_effect(adjusted,t); boot=stratified_bootstrap(adjusted,t,1500,seed+99)
    variance_reduction=1-float(np.var(adjusted,ddof=1))/float(np.var(y,ddof=1)); guard=mean_effect(df.guardrail.to_numpy(),t); decision="ship" if cuped[1]>0 and guard[1]>-0.20 else "hold"
    payload={"project":"ExperimentLab","verification_pass":bool(abs(cuped[0]-effect)<1.0 and variance_reduction>0.30),"scope":"deterministic synthetic randomized-experiment methodology demo","rows":n,"known_simulated_effect":effect,"raw_effect":{"estimate":raw[0],"ci95":[raw[1],raw[2]]},"cuped_effect":{"estimate":cuped[0],"ci95":[cuped[1],cuped[2]],"bootstrap_ci95":list(boot),"theta":theta},"variance_reduction":variance_reduction,"guardrail_effect":{"estimate":guard[0],"ci95":[guard[1],guard[2]],"non_inferiority_margin":-0.20},"power":power_and_mde(float(np.std(adjusted,ddof=1)),n),"decision":decision,"rules":["random assignment","fixed-horizon inference","CUPED uses a pre-treatment covariate","bootstrap resamples within treatment arms","guardrail must not cross the pre-declared harm margin"]}
    df.to_csv(output_dir/"experiment_data.csv",index=False); (output_dir/"verification.json").write_text(json.dumps(payload,indent=2),encoding="utf-8"); print(json.dumps(payload,indent=2)); return payload


def self_test()->None:
    out=run(12000,42,2.5,Path("/tmp/experimentlab_selftest")); assert out["verification_pass"]; assert out["variance_reduction"]>0.30; assert out["cuped_effect"]["ci95"][0]<2.5<out["cuped_effect"]["ci95"][1]; print("ExperimentLab self-test passed.")


def main()->int:
    p=argparse.ArgumentParser(); p.add_argument("--rows",type=int,default=20000); p.add_argument("--seed",type=int,default=42); p.add_argument("--effect",type=float,default=2.5); p.add_argument("--output-dir",type=Path,default=Path("experimentlab_artifacts")); p.add_argument("--self-test",action="store_true"); a=p.parse_args()
    if a.self_test: self_test(); return 0
    r=run(a.rows,a.seed,a.effect,a.output_dir); return 0 if r["verification_pass"] else 1

if __name__=="__main__": raise SystemExit(main())


## Run the real project

The cell below executes the canonical project entry point rather than a rewritten toy version. Keep `RUN_PIPELINE = False` when you only want to inspect the notebook; change it to `True` to reproduce the project.


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    subprocess.run([sys.executable, 'run.py'], cwd=PROJECT, check=True)
else:
    print(f'Reproduce with: cd {PROJECT} && python run.py')


In [ ]:
evidence = []
for folder in (PROJECT / 'results', ROOT / 'verified' / PROJECT_SLUG):
    if folder.exists():
        evidence.extend(sorted(folder.glob('*.json')))
for path in evidence[:5]:
    print('\n---', path.relative_to(ROOT), '---')
    print(path.read_text(encoding='utf-8')[:12000])


## Interview discussion

Be ready to explain the business problem, dataset provenance, cleaning/preprocessing decisions, leakage controls, modelling or analytical choices, evaluation design, limitations, testing strategy and what you would change in production. The key signal is that the notebook, modular source, tests and retained evidence all tell the same story.


## Application engineering layer

The original project above is intentionally preserved. The cells below expose additional canonical Python from this same project—pipelines, APIs, feature code, evaluation, tests, monitoring and other application logic—so the notebook works as a single recruiter-facing project while the modular files remain the production source of truth.


### Canonical source: `src/diagnostics.py`


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Iterable

import numpy as np
import pandas as pd
from scipy.stats import norm


@dataclass(frozen=True)
class EffectEstimate:
    segment: str
    treatment_rows: int
    control_rows: int
    estimate: float
    standard_error: float
    ci_low: float
    ci_high: float


@dataclass(frozen=True)
class BalanceCheck:
    variable: str
    treatment_mean: float
    control_mean: float
    standardized_difference: float
    passed: bool


def standardized_mean_difference(
    treatment_values: np.ndarray,
    control_values: np.ndarray,
) -> float:
    treatment_values = np.asarray(treatment_values, dtype=float)
    control_values = np.asarray(control_values, dtype=float)
    treatment_variance = float(np.var(treatment_values, ddof=1))
    control_variance = float(np.var(control_values, ddof=1))
    pooled_sd = float(np.sqrt((treatment_variance + control_variance) / 2.0))
    if pooled_sd == 0.0:
        return 0.0
    return float((treatment_values.mean() - control_values.mean()) / pooled_sd)


def numeric_balance_table(
    frame: pd.DataFrame,
    treatment_col: str,
    numeric_columns: Iterable[str],
    threshold: float = 0.10,
) -> pd.DataFrame:
    rows: list[dict[str, float | str | bool]] = []
    treatment_mask = frame[treatment_col].to_numpy() == 1
    for column in numeric_columns:
        values = frame[column].to_numpy(dtype=float)
        treated = values[treatment_mask]
        control = values[~treatment_mask]
        smd = standardized_mean_difference(treated, control)
        check = BalanceCheck(
            variable=column,
            treatment_mean=float(treated.mean()),
            control_mean=float(control.mean()),
            standardized_difference=smd,
            passed=bool(abs(smd) < threshold),
        )
        rows.append(asdict(check))
    return pd.DataFrame(rows)


def categorical_balance_table(
    frame: pd.DataFrame,
    treatment_col: str,
    categorical_column: str,
) -> pd.DataFrame:
    counts = (
        frame.groupby([treatment_col, categorical_column], observed=True)
        .size()
        .rename("rows")
        .reset_index()
    )
    totals = counts.groupby(treatment_col)["rows"].transform("sum")
    counts["share"] = counts["rows"] / totals
    pivot = counts.pivot(
        index=categorical_column,
        columns=treatment_col,
        values="share",
    ).fillna(0.0)
    for expected in (0, 1):
        if expected not in pivot.columns:
            pivot[expected] = 0.0
    output = pivot.rename(columns={0: "control_share", 1: "treatment_share"}).reset_index()
    output["absolute_share_gap"] = (
        output["treatment_share"] - output["control_share"]
    ).abs()
    return output.sort_values("absolute_share_gap", ascending=False)


def effect_estimate(
    outcome: np.ndarray,
    treatment: np.ndarray,
    segment: str,
    confidence: float = 0.95,
) -> EffectEstimate:
    outcome = np.asarray(outcome, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    treated = outcome[treatment == 1]
    control = outcome[treatment == 0]
    if len(treated) < 2 or len(control) < 2:
        raise ValueError("Both experiment arms need at least two observations")
    estimate = float(treated.mean() - control.mean())
    standard_error = float(
        np.sqrt(
            treated.var(ddof=1) / len(treated)
            + control.var(ddof=1) / len(control)
        )
    )
    alpha = 1.0 - confidence
    critical = float(norm.ppf(1.0 - alpha / 2.0))
    return EffectEstimate(
        segment=segment,
        treatment_rows=int(len(treated)),
        control_rows=int(len(control)),
        estimate=estimate,
        standard_error=standard_error,
        ci_low=float(estimate - critical * standard_error),
        ci_high=float(estimate + critical * standard_error),
    )


def segment_effect_table(
    frame: pd.DataFrame,
    segment_col: str = "segment",
    treatment_col: str = "treatment",
    outcome_col: str = "outcome",
) -> pd.DataFrame:
    rows: list[dict[str, float | str | int]] = []
    overall = effect_estimate(
        frame[outcome_col].to_numpy(),
        frame[treatment_col].to_numpy(),
        segment="overall",
    )
    rows.append(asdict(overall))
    for segment, group in frame.groupby(segment_col, observed=True):
        estimate = effect_estimate(
            group[outcome_col].to_numpy(),
            group[treatment_col].to_numpy(),
            segment=str(segment),
        )
        rows.append(asdict(estimate))
    return pd.DataFrame(rows)


def bootstrap_effect_distribution(
    outcome: np.ndarray,
    treatment: np.ndarray,
    rounds: int = 2000,
    seed: int = 42,
) -> np.ndarray:
    if rounds < 100:
        raise ValueError("Use at least 100 bootstrap rounds")
    rng = np.random.default_rng(seed)
    outcome = np.asarray(outcome, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    treated_index = np.where(treatment == 1)[0]
    control_index = np.where(treatment == 0)[0]
    draws = np.empty(rounds, dtype=float)
    for index in range(rounds):
        treated_sample = rng.choice(treated_index, size=len(treated_index), replace=True)
        control_sample = rng.choice(control_index, size=len(control_index), replace=True)
        draws[index] = float(
            outcome[treated_sample].mean() - outcome[control_sample].mean()
        )
    return draws


def randomization_inference_pvalue(
    outcome: np.ndarray,
    treatment: np.ndarray,
    permutations: int = 2000,
    seed: int = 42,
) -> float:
    if permutations < 100:
        raise ValueError("Use at least 100 permutations")
    rng = np.random.default_rng(seed)
    outcome = np.asarray(outcome, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    observed = abs(effect_estimate(outcome, treatment, "observed").estimate)
    more_extreme = 0
    for _ in range(permutations):
        shuffled = rng.permutation(treatment)
        candidate = abs(effect_estimate(outcome, shuffled, "permuted").estimate)
        if candidate >= observed:
            more_extreme += 1
    return float((more_extreme + 1) / (permutations + 1))


def bootstrap_summary(draws: np.ndarray) -> dict[str, float]:
    draws = np.asarray(draws, dtype=float)
    return {
        "mean": float(draws.mean()),
        "std": float(draws.std(ddof=1)),
        "ci_low_95": float(np.quantile(draws, 0.025)),
        "ci_high_95": float(np.quantile(draws, 0.975)),
        "probability_positive": float((draws > 0.0).mean()),
    }


def power_curve(
    outcome_sd: float,
    total_sample_sizes: Iterable[int],
    alpha: float = 0.05,
    target_effect: float = 2.5,
) -> pd.DataFrame:
    rows: list[dict[str, float | int]] = []
    critical = float(norm.ppf(1.0 - alpha / 2.0))
    for total_rows in total_sample_sizes:
        per_arm = max(int(total_rows) / 2.0, 2.0)
        standard_error = float(outcome_sd * np.sqrt(2.0 / per_arm))
        noncentrality = float(target_effect / standard_error)
        achieved_power = float(
            1.0
            - norm.cdf(critical - noncentrality)
            + norm.cdf(-critical - noncentrality)
        )
        minimum_detectable_effect = float(
            (critical + norm.ppf(0.80)) * standard_error
        )
        rows.append(
            {
                "total_rows": int(total_rows),
                "target_effect": float(target_effect),
                "achieved_power": achieved_power,
                "mde_at_80pct_power": minimum_detectable_effect,
            }
        )
    return pd.DataFrame(rows)


def guardrail_sensitivity_table(
    guardrail_estimate: float,
    guardrail_ci_low: float,
    effect_ci_low: float,
    margins: Iterable[float] = (-0.05, -0.10, -0.20, -0.30, -0.50),
) -> pd.DataFrame:
    rows: list[dict[str, float | str | bool]] = []
    for margin in margins:
        guardrail_pass = bool(guardrail_ci_low > margin)
        primary_pass = bool(effect_ci_low > 0.0)
        decision = "ship" if guardrail_pass and primary_pass else "hold"
        rows.append(
            {
                "non_inferiority_margin": float(margin),
                "guardrail_estimate": float(guardrail_estimate),
                "guardrail_ci_low": float(guardrail_ci_low),
                "primary_ci_low": float(effect_ci_low),
                "guardrail_pass": guardrail_pass,
                "primary_pass": primary_pass,
                "decision": decision,
            }
        )
    return pd.DataFrame(rows)


def build_diagnostics(
    frame: pd.DataFrame,
    adjusted_outcome: np.ndarray,
    seed: int = 42,
) -> dict[str, object]:
    numeric_balance = numeric_balance_table(
        frame,
        treatment_col="treatment",
        numeric_columns=["pre_metric"],
    )
    categorical_balance = categorical_balance_table(
        frame,
        treatment_col="treatment",
        categorical_column="segment",
    )
    segment_effects = segment_effect_table(frame)
    draws = bootstrap_effect_distribution(
        adjusted_outcome,
        frame["treatment"].to_numpy(),
        rounds=2000,
        seed=seed,
    )
    permutation_pvalue = randomization_inference_pvalue(
        adjusted_outcome,
        frame["treatment"].to_numpy(),
        permutations=1500,
        seed=seed + 1,
    )
    return {
        "numeric_balance": numeric_balance.to_dict(orient="records"),
        "categorical_balance": categorical_balance.to_dict(orient="records"),
        "segment_effects": segment_effects.to_dict(orient="records"),
        "bootstrap": bootstrap_summary(draws),
        "randomization_inference_pvalue": permutation_pvalue,
    }


## Portfolio depth check

**Meaningful code lines currently visible in this notebook:** 330. The portfolio aims for roughly **1,000 meaningful lines** per major project (normally about 800–1,200), and this notebook is below the preferred band and should gain substantive project-specific depth rather than filler.

Line count is not a quality metric by itself. Additional code should only be added when it strengthens the real application: data validation, cleaning, EDA, feature engineering, modelling, tuning, error analysis, explainability, inference, testing, monitoring, APIs, reproducibility or business logic.
